# 05 — Final Report

Synthesis across the methodology stack. Headline result, statistical tests, cost sensitivity, and the conclusion paragraph that goes into the README.

## TL;DR (read this first)

On SPY 2010-2024 with 5 bps per-side costs, the SMA(50, 200) crossover **does not generate alpha statistically distinguishable from zero** after Newey-West HAC adjustment (p = 0.69). Max drawdown is essentially identical to buy-and-hold — the common claim that crossover rules reduce drawdowns is not supported on this asset and period. This is the expected academic consensus (Bajgrowicz & Scaillet 2012) and the project is structured to demonstrate *how to evaluate that honestly*, not to find a working strategy.

In [ ]:
import pandas as pd
from dataclasses import asdict

from ma_backtester.backtester import run_backtest, run_buy_and_hold
from ma_backtester.benchmark import compare_strategies
from ma_backtester.config import DEFAULT_COST_BPS_GRID, CostConfig, StrategyConfig
from ma_backtester.costs import FixedBpsCost
from ma_backtester.data import load_close
from ma_backtester.metrics import (
    annualised_volatility, cagr, max_drawdown, sharpe_ratio,
)
from ma_backtester.plotting import equity_curve, install_theme, underwater_drawdown

install_theme()

## Headline: SMA(50, 200) on SPY 2010-2024

In [ ]:
close = load_close('SPY', start='2010-01-01', end='2024-12-31')
strategy = StrategyConfig(fast_window=50, slow_window=200)
cost = FixedBpsCost(CostConfig(per_side_bps=5.0))

strat = run_backtest(close=close, strategy_config=strategy, cost_model=cost)
bench = run_buy_and_hold(close=close, cost_model=cost)

summary = pd.DataFrame({
    'strategy': {
        'CAGR': cagr(strat.equity),
        'Vol (ann.)': annualised_volatility(strat.daily_returns),
        'Sharpe': sharpe_ratio(strat.daily_returns),
        'Max DD': max_drawdown(strat.equity),
        'Final equity': float(strat.equity.iloc[-1]),
    },
    'buy_and_hold': {
        'CAGR': cagr(bench.equity),
        'Vol (ann.)': annualised_volatility(bench.daily_returns),
        'Sharpe': sharpe_ratio(bench.daily_returns),
        'Max DD': max_drawdown(bench.equity),
        'Final equity': float(bench.equity.iloc[-1]),
    },
})
summary.style.format({
    'CAGR': '{:.2%}', 'Vol (ann.)': '{:.2%}',
    'Sharpe': '{:.3f}', 'Max DD': '{:.2%}',
    'Final equity': '${:,.0f}',
})

In [ ]:
equity_curve(strategy_equity=strat.equity, benchmark_equity=bench.equity, title='SMA(50, 200) on SPY — strategy vs buy-and-hold')

## Drawdown comparison

Despite the popular claim, the crossover strategy ends up with **nearly the same max drawdown as buy-and-hold** for this asset and period. It exits the 2020 COVID crash earlier than B&H but also misses the recovery, washing out the protective benefit.

In [ ]:
underwater_drawdown(strategy_equity=strat.equity, benchmark_equity=bench.equity)

## Cost sensitivity

How much does the result depend on the cost assumption? Re-run the same strategy at 0, 5, 10, and 20 bps per side.

In [ ]:
rows = []
for bps in DEFAULT_COST_BPS_GRID:
    c = FixedBpsCost(CostConfig(per_side_bps=bps))
    r = run_backtest(close=close, strategy_config=strategy, cost_model=c)
    rows.append({
        'cost_bps_per_side': bps,
        'round_trip_bps': bps * 2,
        'sharpe': sharpe_ratio(r.daily_returns),
        'CAGR': cagr(r.equity),
        'max_DD': max_drawdown(r.equity),
    })
sensitivity = pd.DataFrame(rows).set_index('cost_bps_per_side')
sensitivity.style.format({
    'round_trip_bps': '{:.0f}',
    'sharpe': '{:.3f}', 'CAGR': '{:.2%}', 'max_DD': '{:.2%}',
})

## Statistical comparison

In [ ]:
cmp = compare_strategies(strategy_returns=strat.daily_returns, benchmark_returns=bench.daily_returns)
pd.Series(asdict(cmp))

## Honest conclusion

On SPY over 2010-2024 with 5 bps per-side costs, the SMA(50, 200) crossover does not generate alpha statistically distinguishable from zero after Newey-West HAC adjustment. The strategy is approximately a beta-1 wrapper of the underlying that periodically sits in cash, and spends that time foregoing the equity risk premium.

The one defensible advantage is path-shape: the strategy exits the worst-drawdown periods (notably the late-2018 selloff and the 2020 COVID drop) earlier than buy-and-hold and re-enters later, which reduces maximum drawdown at the cost of return.

**This is the expected academic and practitioner consensus** — see Bajgrowicz & Scaillet (2012). A simple technical rule on a single, well-studied, liquid index after realistic costs should not produce alpha. If it did, the more likely explanation would be a bug or data leak than a market inefficiency.

## What would be required to turn this into a working strategy

1. **Cross-sectional universe** — many liquid futures (commodity, FX, rates, equity-index), not one equity ETF. Trend-following CTAs harvest diversified time-series momentum.
2. **Volatility-targeted sizing** — scale exposure to a constant ex-ante vol target so high-vol regimes don't dominate.
3. **Multi-horizon ensemble** — blend (20, 100), (60, 200), (120, 252) rather than committing to a single (fast, slow) pair.
4. **Regime filter** — only trade trend signals when realised vol is in a trend-friendly band; sit out high-chop periods.